# Case study, step 4: manual lemmatization, overlap statistics and heatmaps

Runs without the models. Run from inside `06_casestudy/`, once per keyword (set `keyword` below): Frau, Femizid, Mann, Mord, Opfer, Täter.

**Input:** `table_allneighbours_{keyword}.csv` (100 neighbours per model, 32 self-trained models + cc.de.300), `dict_{keyword}.xlsx` (manual lemmatization dictionary).

**Output:**
- `table_allneighbours_counts_{keyword}.csv`: how often each neighbour occurs (basis for the dictionary)
- `table_allneighbours_pivot_{keyword}.csv`: lemmatized neighbours × models
- `heatmap_overlap_{keyword}.pdf`: number of shared lemmatized neighbours per model pair
- `heatmap_pct_overlap_{keyword}.pdf`: percentage overlap (Frau = Figure 8, Femizid = Figure 9)
- printed statistics reported in the paper (average overlap, words in all / >75% / ≤50% of models)

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from collections import defaultdict
import openpyxl  # needed by pd.read_excel

In [ ]:
# run once per keyword: Frau, Femizid, Mann, Mord, Opfer, Täter
keyword = 'Täter'

In [ ]:
### read in original csv file
df = pd.read_csv(f'table_allneighbours_{keyword}.csv')

In [ ]:
### create word count 

counts = (
    df["Neighbor"]
    .value_counts()
    .reset_index()
)
counts.columns = ["Neighbor", "Count"]

counts.to_csv(f"table_allneighbours_counts_{keyword}.csv", index=False)

In [ ]:
### manual lemmatization of nearest neighbours 
dict_df = pd.read_excel(f"dict_{keyword}.xlsx", header=0)
dict_df.columns = ["Neighbor", "Count", "Lemma", "Count_Lemma"]

In [ ]:
### lemmatize with dictionary
# --- Build lemma mapping ---
# If Lemma is empty, keep original word; otherwise use the lemma
dict_df["Resolved"] = dict_df["Lemma"].fillna(dict_df["Neighbor"]).str.strip()
lemma_map = dict_df.set_index("Neighbor")["Resolved"].to_dict()

# --- Apply lemmatization to original dataframe ---
df["Neighbor_lemma"] = df["Neighbor"].map(lemma_map).fillna(df["Neighbor"])

In [ ]:
### make a pivot table
pivot = (
    df.groupby(["Neighbor_lemma", "Model"])
    .size()
    .unstack(fill_value=0)
)
pivot.index.name = "Neighbor"

pivot.to_csv(f"table_allneighbours_pivot_{keyword}.csv")

In [ ]:
### and a heatmap
def short_labels(labels):
    return ['_'.join(label.split('_')[:2]) for label in labels]

def plot_heatmap(overlap_matrix, filename):
    plt.figure(figsize=(20, 16))
    cmap = mcolors.LinearSegmentedColormap.from_list("custom_blue", ["#ffffff", "#0063A6"])
    sns.heatmap(
        overlap_matrix,
        annot=True,
        fmt=".0f",
        cmap=cmap,
        xticklabels=short_labels(overlap_matrix.columns),
        yticklabels=short_labels(overlap_matrix.index),
    )
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.savefig(filename, format="pdf")
    plt.show()

# --- Binarize ---
binary = (pivot > 0).astype(int)

# --- Compute overlap matrix ---
overlap = binary.T.dot(binary)  # models x models

# --- Plot heatmap ---
plot_heatmap(overlap, f"heatmap_overlap_{keyword}.pdf")

In [ ]:
### get some stats 
# ── Stats functions ──────────────────────────────────────────────────────────

def calculate_percentage_overlap_and_average(df):
    """Percentage of col1's words that also appear in col2, for every pair."""
    overlap_matrix = pd.DataFrame(index=df.columns, columns=df.columns, dtype=float)
    total_overlap = 0
    num_pairs = 0

    for col1 in df.columns:
        for col2 in df.columns:
            set1 = set(df[col1].dropna())
            set2 = set(df[col2].dropna())
            overlap = len(set1.intersection(set2))
            percentage_overlap = (overlap / len(set1)) * 100 if set1 else 0
            overlap_matrix.loc[col1, col2] = percentage_overlap
            if col1 != col2:
                total_overlap += percentage_overlap
                num_pairs += 1

    average_overlap = total_overlap / num_pairs if num_pairs > 0 else 0
    print(f"Average overlap: {average_overlap:.2f}%")
    return overlap_matrix


def find_words_in_all_columns(words_table):
    """Map each word to the set of models it appears in."""
    word_locations = defaultdict(set)
    for col in words_table.columns:
        for word in words_table[col].dropna():
            word_locations[word].add(col)

    words_by_column_count = defaultdict(list)
    for word, columns in word_locations.items():
        words_by_column_count[len(columns)].append((word, columns))

    return words_by_column_count


# ── Run ──────────────────────────────────────────────────────────────────────

lemmatized_df = pd.DataFrame(
    {col: pd.Series(binary.index[binary[col] == 1].tolist()) for col in binary.columns}
)

# Word distribution across models
words_in_all_columns = find_words_in_all_columns(lemmatized_df)
total_words = 0

for column_count in sorted(words_in_all_columns.keys(), reverse=True):
    words_list = words_in_all_columns[column_count]
    total_words += len(words_list)
    print(f"Words in {column_count} model(s) (Total: {len(words_list)}):")
    for word, columns in words_list:
        print(f"  '{word}' → {', '.join(sorted(columns))}")
    print()

print(f"Total unique words for '{keyword}': {total_words}")

# Words in half the models or fewer
n_models = len(lemmatized_df.columns)
half_columns = n_models // 2
words_in_half_or_less = sum(
    len(words_in_all_columns[c]) for c in range(1, half_columns + 1)
)
print(f"Words in ≤50% of models ({half_columns}): {words_in_half_or_less}")

# Words in more than 75% of models
seventy_five_pct = n_models * 0.75
words_in_75pct_or_more = sum(
    len(words_in_all_columns[c]) for c in range(int(seventy_five_pct) + 1, n_models + 1)
)
print(f"Words in >75% of models (>{int(seventy_five_pct)}): {words_in_75pct_or_more}")

# Percentage overlap matrix
pct_overlap_matrix = calculate_percentage_overlap_and_average(lemmatized_df)
plot_heatmap(pct_overlap_matrix, f"heatmap_pct_overlap_{keyword}.pdf")